# Policy Grounding Agent\n\nThis notebook builds a **Policy Research & Quality Review Agent** that uses HERALD as a live validation service.\n\nThe agent:\n1. Takes a policy document (or a snippet) and a set of research questions\n2. Generates claims / answers for each question using an LLM\n3. Calls the HERALD MCP server to validate each claim against the source\n4. Produces an annotated report showing verdict + tier routing in real time\n\n**Prerequisites:** HERALD MCP server must be running:\n```bash\nuv run herald-mcp\n```\n\n> **Note:** This notebook is a prototype. The final agent will live at `src/herald/agent/policy_agent.py`."

## 1. Imports & Config"

In [1]:
import json
import os
import textwrap
from dataclasses import dataclass, field
from pathlib import Path

import httpx
from dotenv import load_dotenv
from google import genai

load_dotenv(Path("../.env"))

# ── HERALD MCP server ─────────────────────────────────────────────────────────
HERALD_MCP_URL = "http://localhost:8000/mcp"

# ── LLM (Gemini, same provider as the rest of HERALD) ────────────────────────
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
client_genai = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-2.0-flash"

print(f"HERALD MCP: {HERALD_MCP_URL}")
print(f"LLM model:  {GEMINI_MODEL}")

HERALD MCP: http://localhost:8000/mcp
LLM model:  gemini-2.0-flash


## 2. HERALD MCP Client\n\nA thin wrapper that speaks the MCP JSON-RPC protocol over HTTP.\nEach tool call is a `POST /mcp` with `method: \"tools/call\"`."

In [2]:
@dataclass
class ValidationResult:
    claim: str
    question: str
    verdict: str  # valid / invalid / uncertain
    resolved_at_tier: int
    confidence: float | None = None
    reasoning: str | None = None
    tier_detail: dict = field(default_factory=dict)
    error: str | None = None


def _mcp_initialize(client: httpx.Client) -> str:
    """Send MCP initialize handshake, return session-id header value."""
    payload = {
        "jsonrpc": "2.0",
        "id": 0,
        "method": "initialize",
        "params": {
            "protocolVersion": "2025-03-26",
            "capabilities": {},
            "clientInfo": {"name": "policy-agent-notebook", "version": "0.1"},
        },
    }
    r = client.post(HERALD_MCP_URL, json=payload)
    r.raise_for_status()
    session_id = r.headers.get("mcp-session-id", "")
    # send initialized notification
    client.post(
        HERALD_MCP_URL,
        json={"jsonrpc": "2.0", "method": "notifications/initialized", "params": {}},
        headers={"mcp-session-id": session_id} if session_id else {},
    )
    return session_id


def herald_validate(
    claim: str,
    source_context: str,
    checkpoint_type: str = "claim_extraction",
    query: str = "",
    question: str = "",
) -> ValidationResult:
    """Call HERALD validate_checkpoint via MCP and return a structured result."""
    with httpx.Client(timeout=120) as client:
        session_id = _mcp_initialize(client)
        headers = {"mcp-session-id": session_id} if session_id else {}

        payload = {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {
                "name": "validate_checkpoint",
                "arguments": {
                    "output_text": claim,
                    "source_context": source_context,
                    "checkpoint_type": checkpoint_type,
                    "query": query,
                },
            },
        }
        r = client.post(HERALD_MCP_URL, json=payload, headers=headers)
        r.raise_for_status()
        body = r.json()

    if "error" in body:
        return ValidationResult(
            claim=claim,
            question=question,
            verdict="error",
            resolved_at_tier=-1,
            error=str(body["error"]),
        )

    # MCP wraps the tool return value in result.content[0].text (JSON string)
    raw = body.get("result", {})
    content_list = raw.get("content", [])
    if content_list and isinstance(content_list[0], dict):
        data = json.loads(content_list[0].get("text", "{}"))
    else:
        data = raw  # fallback if server returns unwrapped dict

    if "error" in data:
        return ValidationResult(
            claim=claim,
            question=question,
            verdict="error",
            resolved_at_tier=-1,
            error=data["error"],
        )

    # Pull confidence from the tier that resolved
    tier_key = f"tier{data.get('resolved_at_tier', 1)}"
    tier_data = data.get(tier_key, {})

    return ValidationResult(
        claim=claim,
        question=question,
        verdict=data.get("verdict", "uncertain"),
        resolved_at_tier=data.get("resolved_at_tier", -1),
        confidence=tier_data.get("confidence"),
        reasoning=tier_data.get("reasoning"),
        tier_detail={k: v for k, v in data.items() if k.startswith("tier")},
    )


print("MCP client ready.")

MCP client ready.


## 3. Policy Agent\n\nThe agent loop:\n1. For each question, call Gemini to generate a claim grounded in the policy document\n2. Immediately call HERALD to validate the claim\n3. Print live status as each result comes in"

In [3]:
VERDICT_ICON = {"valid": "✓", "invalid": "✗", "uncertain": "?", "error": "!"}
TIER_LABEL = {1: "T1·NLI", 2: "T2·Judge", 3: "T3·Debate", 4: "T4·Human"}

CLAIM_PROMPT = """\
You are a careful policy analyst. Given the policy excerpt below, answer the question \
in ONE concise sentence. Your answer must be directly grounded in the excerpt — do not \
add information from outside it. If the excerpt does not contain enough information to \
answer, say so explicitly.

Policy excerpt:
{source}

Question: {question}

Answer (one sentence):"""


def generate_claim(source_context: str, question: str) -> str:
    response = client_genai.models.generate_content(
        model=GEMINI_MODEL,
        contents=CLAIM_PROMPT.format(source=source_context, question=question),
    )
    return response.text.strip()


def _print_result(result: ValidationResult, idx: int, total: int) -> None:
    icon = VERDICT_ICON.get(result.verdict, "?")
    tier = TIER_LABEL.get(result.resolved_at_tier, f"T{result.resolved_at_tier}")
    conf = f"{result.confidence:.2f}" if result.confidence is not None else "n/a"
    print(f"\n[{idx}/{total}] {icon} {result.verdict.upper():10s}  tier={tier}  conf={conf}")
    print(f"  Q: {result.question}")
    print(f"  Claim: {textwrap.fill(result.claim, width=90, subsequent_indent='         ')}")
    if result.reasoning:
        print(
            f"  Reason: {textwrap.fill(result.reasoning[:300], width=90, subsequent_indent='          ')}"
        )
    if result.error:
        print(f"  ERROR: {result.error}")


def run_policy_agent(
    policy_doc: str,
    questions: list[str],
    checkpoint_type: str = "claim_extraction",
) -> list[ValidationResult]:
    """
    For each question:
      1. Generate a claim from the policy doc via LLM
      2. Validate the claim via HERALD MCP
      3. Print live result
    Returns a list of ValidationResult for downstream analysis.
    """
    results = []
    total = len(questions)
    print(f"Running policy agent on {total} questions...\n{'─' * 70}")

    for i, question in enumerate(questions, 1):
        print(f"[{i}/{total}] Generating claim for: {question[:80]}...")
        claim = generate_claim(policy_doc, question)

        print("         Validating via HERALD...", end=" ", flush=True)
        result = herald_validate(
            claim=claim,
            source_context=policy_doc,
            checkpoint_type=checkpoint_type,
            query=question,
            question=question,
        )
        _print_result(result, i, total)
        results.append(result)

    print(f"\n{'─' * 70}")
    valid = sum(1 for r in results if r.verdict == "valid")
    invalid = sum(1 for r in results if r.verdict == "invalid")
    unc = sum(1 for r in results if r.verdict == "uncertain")
    print(f"Summary: {valid} valid  |  {invalid} invalid  |  {unc} uncertain  (out of {total})")
    return results


print("Agent defined.")

Agent defined.


## 4. Demo: GSA Federal Supply Schedule Policy\n\nUsing a real excerpt from our gov_report dataset — the GSA MAS procurement policy."

In [4]:
# Load a real policy excerpt from our dataset
with open("../data/test_sets/gov_report_v2_filtered.json") as f:
    cases = json.load(f)

# Pick the first case with a substantial source_context as our policy doc
sample = next(c for c in cases if len(c["source_context"]) > 400)
POLICY_DOC = sample["source_context"]

print("Policy excerpt (first 600 chars):")
print("─" * 70)
print(POLICY_DOC[:600])
print("─" * 70)
print(f"\nCheckpoint type in dataset: {sample['checkpoint_type']}")

Policy excerpt (first 600 chars):
──────────────────────────────────────────────────────────────────────
GSA, through its Federal Supply Schedule (FSS) program, makes commonly used commercial items and services available to federal agencies. MAS, the largest FSS program, is designed to provide federal agencies with a simplified method for acquiring varying quantities of a wide range of commercially available goods and services, such as office furniture and supplies, personal computers, scientific equipment, library services, network support, and laboratory testing services. The MAS program provides several advantages to both federal agencies and vendors. Agencies can use a simplified method of ac
──────────────────────────────────────────────────────────────────────

Checkpoint type in dataset: claim_extraction


In [5]:
# Research questions a policy analyst might ask
QUESTIONS = [
    "What is the primary purpose of the MAS program?",
    "What types of goods or services can federal agencies purchase through MAS?",
    "What advantages does the MAS program provide to federal agencies?",
    "What advantages does the MAS program provide to vendors?",
    "Does the MAS program require competitive bidding for every purchase?",  # likely uncertain/invalid — tests HERALD
]

results = run_policy_agent(POLICY_DOC, QUESTIONS, checkpoint_type="claim_extraction")

Running policy agent on 5 questions...
──────────────────────────────────────────────────────────────────────
[1/5] Generating claim for: What is the primary purpose of the MAS program?...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 44.209699991s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go

## 5. Tier Routing Visualization\n\nSee which tier resolved each claim — this is what makes HERALD's escalation visible."

In [ ]:
import matplotlib.pyplot as plt

VERDICT_COLOR = {
    "valid": "#2ecc71",
    "invalid": "#e74c3c",
    "uncertain": "#f39c12",
    "error": "#95a5a6",
}
TIER_NAMES = {1: "Tier 1\nNLI", 2: "Tier 2\nLLM Judge", 3: "Tier 3\nDebate", 4: "Tier 4\nHuman"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Policy Agent — HERALD Validation Report", fontsize=14, fontweight="bold")

# ── Left: tier routing timeline ───────────────────────────────────────────────
ax = axes[0]
ax.set_title("Tier Routing per Claim", fontweight="bold")

for i, r in enumerate(results):
    y = len(results) - i
    color = VERDICT_COLOR.get(r.verdict, "#95a5a6")
    tier = r.resolved_at_tier if r.resolved_at_tier > 0 else 1

    # Draw pipeline path (grey dotted)
    for t in range(1, tier):
        ax.plot([t, t + 1], [y, y], color="#bdc3c7", linewidth=1.5, linestyle=":")

    # Draw resolved node
    ax.scatter([tier], [y], color=color, s=200, zorder=5)
    ax.text(
        tier + 0.1, y, f" {VERDICT_ICON.get(r.verdict, '?')} {r.verdict}", va="center", fontsize=9
    )

    # Question label on y-axis
    label = r.question[:45] + "…" if len(r.question) > 45 else r.question
    ax.text(0.85, y, label, ha="right", va="center", fontsize=8, color="#555")

ax.set_xlim(0.8, 4.5)
ax.set_ylim(0, len(results) + 1)
ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels([TIER_NAMES.get(t, f"T{t}") for t in [1, 2, 3, 4]], fontsize=9)
ax.set_yticks([])
ax.spines[["left", "right", "top"]].set_visible(False)

# ── Right: verdict distribution ───────────────────────────────────────────────
ax2 = axes[1]
ax2.set_title("Verdict Distribution", fontweight="bold")

verdict_counts = {}
for r in results:
    verdict_counts[r.verdict] = verdict_counts.get(r.verdict, 0) + 1

labels = list(verdict_counts.keys())
values = list(verdict_counts.values())
colors = [VERDICT_COLOR.get(label, "#95a5a6") for label in labels]
wedges, texts, autotexts = ax2.pie(
    values,
    labels=labels,
    colors=colors,
    autopct="%1.0f%%",
    startangle=90,
    textprops={"fontsize": 11},
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight("bold")

plt.tight_layout()
plt.savefig("../results/plots/policy_agent_demo.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to results/plots/policy_agent_demo.png")

## 6. Full Report Table\n\nClaim-by-claim breakdown with tier reasoning — what you'd hand to a human reviewer."

In [ ]:
import pandas as pd

rows = []
for r in results:
    rows.append(
        {
            "Question": r.question,
            "Claim (generated)": r.claim,
            "Verdict": r.verdict,
            "Resolved at": TIER_LABEL.get(r.resolved_at_tier, f"T{r.resolved_at_tier}"),
            "Confidence": f"{r.confidence:.2f}" if r.confidence else "—",
            "Reasoning (excerpt)": (r.reasoning or "")[:200],
        }
    )

df = pd.DataFrame(rows)


def color_verdict(val):
    colors = {
        "valid": "background-color: #d5f5e3",
        "invalid": "background-color: #fadbd8",
        "uncertain": "background-color: #fef9e7",
    }
    return colors.get(val, "")


df.style.applymap(color_verdict, subset=["Verdict"]).set_properties(**{"text-align": "left"})

## 7. Try Your Own Policy Document\n\nDrop any policy text and questions here to run the agent on your own material."

In [ ]:
MY_POLICY = """
Paste your policy excerpt here.
"""

MY_QUESTIONS = [
    "What is the main obligation established by this policy?",
    "Who is responsible for compliance?",
    "What are the penalties for non-compliance?",
]

# Uncomment to run:
# my_results = run_policy_agent(MY_POLICY.strip(), MY_QUESTIONS, checkpoint_type="claim_extraction")